In [ ]:
# @title Imports & Configuration
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForMaskedLM, 
    DataCollatorForLanguageModeling, 
    Trainer, 
    TrainingArguments
)
from datasets import Dataset

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

@dataclass
class Config:
    # Paths (Update these based on your Drive location)
    product_path: str = "/content/products.parquet" 
    query_path: str = "/content/queries.parquet"
    output_dir: str = "./persian_ecommerce_mlm"
    
    # Model
    base_model_id: str = "HooshvareLab/distilbert-fa-zwnj-base"
    
    # Data Mixing
    query_ratio: float = 0.5    # 50% queries, 50% products
    random_seed: int = 42
    test_size: float = 0.1      # 10% for validation
    
    # Tokenization
    max_length: int = 128       # Max sequence length
    mlm_probability: float = 0.15 # Percentage of tokens to mask
    
    # Training Hyperparameters
    batch_size: int = 32
    learning_rate: float = 2e-5
    num_train_epochs: int = 3
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    save_total_limit: int = 2   # Only keep the last 2 checkpoints
    logging_steps: int = 100
    
config = Config()
print("Configuration loaded.")

In [ ]:
# @title Load & Balance Data
def load_and_mix_data(config):
    print("Loading datasets...")
    # Load Parquet files
    # Note: Ensure files are uploaded to Colab or mounted via Drive
    try:
        df_products = pd.read_parquet(config.product_path)
        df_queries = pd.read_parquet(config.query_path)
    except FileNotFoundError:
        print("Error: Parquet files not found. Please upload 'products.parquet' and 'queries.parquet'.")
        return None

    # Extract clean text lists
    # Drop NAs to prevent errors
    products_text = df_products['cleaned_text'].dropna().astype(str).tolist()
    queries_text = df_queries['cleaned_query'].dropna().astype(str).tolist()

    print(f"Original counts - Products: {len(products_text)}, Queries: {len(queries_text)}")

    # Undersampling Logic
    # We determine the target size based on the smaller dataset to ensure we can meet the ratio
    # If we want 50/50, we take min(len_p, len_q) from both.
    
    n_products = len(products_text)
    n_queries = len(queries_text)
    
    # Simple logic for 50/50 undersampling:
    # Take the minimum length and sample that amount from both
    if config.query_ratio == 0.5:
        sample_size = min(n_products, n_queries)
        
        # Sample
        rng = np.random.default_rng(config.random_seed)
        selected_products = rng.choice(products_text, size=sample_size, replace=False)
        selected_queries = rng.choice(queries_text, size=sample_size, replace=False)
    else:
        # Complex logic if ratio is not 0.5 (optional implementation)
        # For now, sticking to the prompt's main request of undersampling for balance
        sample_size = min(n_products, n_queries)
        rng = np.random.default_rng(config.random_seed)
        selected_products = rng.choice(products_text, size=sample_size, replace=False)
        selected_queries = rng.choice(queries_text, size=sample_size, replace=False)

    print(f"Undersampled counts - Products: {len(selected_products)}, Queries: {len(selected_queries)}")
    
    # Combine
    combined_texts = np.concatenate([selected_products, selected_queries])
    np.random.shuffle(combined_texts) # Shuffle to mix batches
    
    # Create HF Dataset
    dataset = Dataset.from_dict({'text': combined_texts})
    return dataset

dataset = load_and_mix_data(config)
dataset

In [ ]:
# @title Tokenization
tokenizer = AutoTokenizer.from_pretrained(config.base_model_id)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=config.max_length,
        return_special_tokens_mask=True
    )

print("Tokenizing dataset (this may take a moment)...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

# Split Train/Val
split_dataset = tokenized_datasets.train_test_split(
    test_size=config.test_size, 
    seed=config.random_seed
)

print(f"Train size: {len(split_dataset['train'])}")
print(f"Validation size: {len(split_dataset['test'])}")

In [ ]:
# @title Initialize Model & Trainer
model = AutoModelForMaskedLM.from_pretrained(config.base_model_id)

# Data Collator handles the masking dynamically
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=config.mlm_probability
)

training_args = TrainingArguments(
    output_dir=config.output_dir,
    overwrite_output_dir=True,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    evaluation_strategy="epoch", # Evaluate at end of epoch
    save_strategy="epoch",       # Save at end of epoch
    load_best_model_at_end=True, # Always return the best model
    metric_for_best_model="eval_loss",
    logging_steps=config.logging_steps,
    seed=config.random_seed,
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU available
    report_to="none" # Disable wandb/mlflow for simple colab usage
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    data_collator=data_collator,
)

print("Trainer initialized.")

In [ ]:
# @title Run Training
print("Starting training...")
train_result = trainer.train()

print(f"Training completed. Best model loaded from checkpoint.")
print(train_result.metrics)

In [ ]:
# @title Plot Training Trends
def plot_history(trainer_obj):
    history = trainer_obj.state.log_history
    
    # Extract data
    train_loss = []
    eval_loss = []
    train_steps = []
    eval_steps = []
    
    for entry in history:
        if 'loss' in entry:
            train_loss.append(entry['loss'])
            train_steps.append(entry['step'])
        if 'eval_loss' in entry:
            eval_loss.append(entry['eval_loss'])
            # Eval usually happens at specific steps/epochs
            eval_steps.append(entry['step'])
            
    plt.figure(figsize=(10, 6))
    plt.plot(train_steps, train_loss, label='Training Loss', alpha=0.6)
    
    # Plot eval loss as markers since they are less frequent
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red', marker='o', linestyle='--')
        
    plt.title(f'MLM Domain Adaptation: {config.base_model_id}')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_history(trainer)

In [ ]:
# @title Save Best Model
save_path = f"{config.output_dir}/best_domain_adapted"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to: {save_path}")

# Optional: Zip for download if using Colab
import shutil
shutil.make_archive("domain_adapted_model", 'zip', save_path)
print("Zipped model available as 'domain_adapted_model.zip'")